# Open Notebook & Additional Resources

<a target="_blank" href="https://colab.research.google.com/github/Nicolepcx/ORM_AI_Agents_Bootcamp/blob/main/hands_on/DAY_1_HANDS_ON_SESSION_5_communication_patterns_exchange.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>
<a target="_blank" href="https://learning.oreilly.com/library/view/ai-agents-the/0642572247775/">
  <img src="https://img.shields.io/badge/AI%20Agents%20Book-Read%20on%20O'Reilly-d40101?style=flat" alt="AI Agents Book – Read on O'Reilly"/>
</a>





<font color="red" size="10">
<b>HANDS-ON TIME: 15 mins</b>
</font>

# Exchange a Communication Pattern

## Exercise Goal

Keep the same MAS topology (peer-to-peer team with three specialist agents), but swap the communication protocol and observe how coordination changes.

- Topology (fixed): `researcher`, `planner`, `writer`
- Protocols (swappable):
  - `shared_memory` (blackboard)
  - `event_driven` (publish-subscribe)

## What this demonstrates

- Collaboration pattern = who works with whom.
- Communication protocol = how information flows.
- Swapping only the protocol changes context pressure, consistency risk, and traceability.

In [1]:
# Install minimal dependencies for this notebook.
!pip install -q "langchain-openai" "python-dotenv>=1.0,<2.0" "pydantic>=2,<3"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 4.7 MB/s eta 0:00:00


In [2]:
from __future__ import annotations

import json
import os
import uuid
from datetime import datetime, timezone
from typing import Any, Literal

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-nano")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY is required. Set it in your environment or .env file.")

LLM = ChatOpenAI(model=OPENAI_MODEL, api_key=OPENAI_API_KEY, temperature=0)
print(f"Using model: {OPENAI_MODEL}")


def new_id(prefix: str) -> str:
    return f"{prefix}_{uuid.uuid4().hex[:8]}"


def now_ts() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def jdump(x: Any) -> str:
    if isinstance(x, BaseModel):
        return x.model_dump_json(indent=2)
    return json.dumps(x, indent=2, default=str)


def payload_chars(x: Any) -> int:
    return len(jdump(x))

Using model: gpt-5.4-nano


## Shared Topology

We keep the same three-agent collaboration pattern in both protocol runs:

1. `researcher` gathers key facts and uncertainty
2. `planner` turns facts into an actionable plan
3. `writer` produces final output for a stakeholder

Only the communication layer changes.

In [3]:
TASK_LIBRARY: dict[str, dict[str, Any]] = {
    "creative_writing": {
        "objective": "Draft a short campaign narrative for a climate-tech startup launch.",
        "constraints": [
            "Use a persuasive but concrete tone.",
            "Keep it under 180 words.",
            "Include one measurable outcome.",
        ],
        "acceptance_criteria": [
            "Narrative is coherent and specific.",
            "No unsupported claims.",
        ],
    },
    "strategic_planning": {
        "objective": "Propose a 30-day rollout plan for an internal AI assistant.",
        "constraints": [
            "Max 5 plan steps.",
            "Must include risk controls.",
            "Assume limited engineering bandwidth.",
        ],
        "acceptance_criteria": [
            "Plan is executable.",
            "Dependencies and risks are explicit.",
        ],
    },
    "research_synthesis": {
        "objective": "Synthesize trade-offs between open-source and managed vector databases.",
        "constraints": [
            "Focus on cost, reliability, and operational complexity.",
            "Do not assume hidden benchmarks.",
            "Keep synthesis under 220 words.",
        ],
        "acceptance_criteria": [
            "Balanced comparison.",
            "Decision guidance included.",
        ],
    },
}


class WorkBrief(BaseModel):
    work_id: str
    task_type: Literal["creative_writing", "strategic_planning", "research_synthesis"]
    objective: str
    constraints: list[str]
    acceptance_criteria: list[str]


class ResearchOutput(BaseModel):
    key_findings: list[str]
    uncertainties: list[str]
    source_quality: Literal["low", "medium", "high"]


class PlanOutput(BaseModel):
    strategy_steps: list[str]
    risks: list[str]
    success_signal: str


class WriterOutput(BaseModel):
    final_text: str
    checklist: list[str]


def make_brief(task_type: str) -> WorkBrief:
    task = TASK_LIBRARY[task_type]
    return WorkBrief(
        work_id=new_id("work"),
        task_type=task_type,
        objective=task["objective"],
        constraints=task["constraints"],
        acceptance_criteria=task["acceptance_criteria"],
    )

## Protocol A: Shared Memory (Blackboard)

All agents read and write one central board. This is easy to audit but can grow quickly and create stale-write risk.

In [4]:
class BlackboardEntry(BaseModel):
    entry_id: str
    author: str
    kind: Literal["research", "plan", "draft", "note"]
    content: dict[str, Any]
    ts: str


class BlackboardState(BaseModel):
    brief: WorkBrief
    entries: list[BlackboardEntry] = Field(default_factory=list)
    revision: int = 0


def run_shared_memory(brief: WorkBrief) -> dict[str, Any]:
    researcher = LLM.with_structured_output(ResearchOutput)
    planner = LLM.with_structured_output(PlanOutput)
    writer = LLM.with_structured_output(WriterOutput)

    board = BlackboardState(brief=brief)
    context_log: list[dict[str, Any]] = []

    researcher_context = {
        "brief": brief.model_dump(),
        "board": board.model_dump(),
    }
    r = researcher.invoke(
        f"You are the researcher. Return structured output only.\n\nContext:\n{jdump(researcher_context)}"
    )
    board.entries.append(
        BlackboardEntry(
            entry_id=new_id("bb"),
            author="researcher",
            kind="research",
            content=r.model_dump(),
            ts=now_ts(),
        )
    )
    board.revision += 1
    context_log.append({"agent": "researcher", "chars": payload_chars(researcher_context)})

    planner_context = {
        "brief": brief.model_dump(),
        "board": board.model_dump(),
    }
    p = planner.invoke(
        f"You are the planner. Use the latest research from the board. Return structured output only.\n\nContext:\n{jdump(planner_context)}"
    )
    board.entries.append(
        BlackboardEntry(
            entry_id=new_id("bb"),
            author="planner",
            kind="plan",
            content=p.model_dump(),
            ts=now_ts(),
        )
    )
    board.revision += 1
    context_log.append({"agent": "planner", "chars": payload_chars(planner_context)})

    writer_context = {
        "brief": brief.model_dump(),
        "board": board.model_dump(),
    }
    w = writer.invoke(
        f"You are the writer. Build final output from the board content. Return structured output only.\n\nContext:\n{jdump(writer_context)}"
    )
    board.entries.append(
        BlackboardEntry(
            entry_id=new_id("bb"),
            author="writer",
            kind="draft",
            content=w.model_dump(),
            ts=now_ts(),
        )
    )
    board.revision += 1
    context_log.append({"agent": "writer", "chars": payload_chars(writer_context)})

    return {
        "protocol": "shared_memory",
        "board": board,
        "final": w,
        "context_log": context_log,
        "total_context_chars": sum(item["chars"] for item in context_log),
        "consistency_risk": "medium-high (stale writes possible on shared board)",
        "traceability": "high (single board audit)",
    }

## Protocol B: Event-Driven

Agents publish compact events. Subscribers react only to events they care about. This reduces shared-state exposure but can fail silently if routing/subscriptions are wrong.

In [5]:
class EventEnvelope(BaseModel):
    event_id: str
    event_type: Literal["research_ready", "plan_ready", "draft_ready"]
    producer: str
    payload: dict[str, Any]
    correlation_id: str
    ts: str


def run_event_driven(brief: WorkBrief) -> dict[str, Any]:
    researcher = LLM.with_structured_output(ResearchOutput)
    planner = LLM.with_structured_output(PlanOutput)
    writer = LLM.with_structured_output(WriterOutput)

    correlation_id = new_id("corr")
    event_log: list[EventEnvelope] = []
    context_log: list[dict[str, Any]] = []

    researcher_context = {"brief": brief.model_dump()}
    r = researcher.invoke(
        f"You are the researcher. Return structured output only.\n\nContext:\n{jdump(researcher_context)}"
    )
    ev1 = EventEnvelope(
        event_id=new_id("evt"),
        event_type="research_ready",
        producer="researcher",
        payload=r.model_dump(),
        correlation_id=correlation_id,
        ts=now_ts(),
    )
    event_log.append(ev1)
    context_log.append({"agent": "researcher", "chars": payload_chars(researcher_context)})

    planner_context = {
        "brief": brief.model_dump(),
        "event": ev1.model_dump(),
    }
    p = planner.invoke(
        f"You are the planner. Consume the research_ready event and return structured output only.\n\nContext:\n{jdump(planner_context)}"
    )
    ev2 = EventEnvelope(
        event_id=new_id("evt"),
        event_type="plan_ready",
        producer="planner",
        payload=p.model_dump(),
        correlation_id=correlation_id,
        ts=now_ts(),
    )
    event_log.append(ev2)
    context_log.append({"agent": "planner", "chars": payload_chars(planner_context)})

    writer_context = {
        "brief": brief.model_dump(),
        "research_event": ev1.model_dump(),
        "plan_event": ev2.model_dump(),
    }
    w = writer.invoke(
        f"You are the writer. Consume event payloads and return structured output only.\n\nContext:\n{jdump(writer_context)}"
    )
    ev3 = EventEnvelope(
        event_id=new_id("evt"),
        event_type="draft_ready",
        producer="writer",
        payload=w.model_dump(),
        correlation_id=correlation_id,
        ts=now_ts(),
    )
    event_log.append(ev3)
    context_log.append({"agent": "writer", "chars": payload_chars(writer_context)})

    return {
        "protocol": "event_driven",
        "event_log": event_log,
        "final": w,
        "context_log": context_log,
        "total_context_chars": sum(item["chars"] for item in context_log),
        "consistency_risk": "medium (routing errors more likely than stale overwrite)",
        "traceability": "medium-high (good if event IDs and correlation IDs are enforced)",
    }

## Run Both Protocols on the Same Task

In [6]:
TASK_TYPE = "strategic_planning"  # change to creative_writing or research_synthesis
brief = make_brief(TASK_TYPE)

shared_result = run_shared_memory(brief)
event_result = run_event_driven(brief)

print("=== Shared memory final output ===")
print(jdump(shared_result["final"]))

print("\n=== Event-driven final output ===")
print(jdump(event_result["final"]))

print("\n=== Coordination comparison ===")
print(f"shared_memory total context chars: {shared_result['total_context_chars']}")
print(f"event_driven total context chars:  {event_result['total_context_chars']}")
print(f"shared_memory consistency risk:    {shared_result['consistency_risk']}")
print(f"event_driven consistency risk:     {event_result['consistency_risk']}")
print(f"shared_memory traceability:        {shared_result['traceability']}")
print(f"event_driven traceability:         {event_result['traceability']}")

=== Shared memory final output ===
{
  "final_text": "30-Day Rollout Plan (Internal AI Assistant) — Max 5 Steps | Limited Engineering Bandwidth\n\nStep 1 (Days 1–7): Choose narrow pilot + lock prerequisites\n- Select 1–2 high-impact workflows (assumption examples: internal doc Q&A; ticket summarization).\n- Define measurable success metrics (e.g., accuracy/utility threshold, % responses with citations, human approval rate).\n- Confirm explicit dependencies: approved LLM/chat service, Identity/SSO, logging/auditing, knowledge sources (existing KB/docs), and legal/security approvals.\n- Set minimal “model/tool scope” for Weeks 1–2: read-only KB retrieval; no external actions.\n\nStep 2 (Days 8–14): Build smallest viable assistant (workflow-based) + risk controls\n- Implement guided assistant flow: intake → retrieve from KB → draft answer with citations → ask clarifying questions when needed.\n- Add risk controls: human-in-the-loop confirmation for initial outputs; conservative guardrails

## Optional: Force Failure Modes

This cell intentionally introduces anti-patterns so you can see where production systems collapse.

In [7]:
def inject_shared_memory_failure(shared: dict[str, Any]) -> dict[str, Any]:
    board = shared["board"].model_copy(deep=True)
    noise = {
        "history": ["old update"] * 120,
        "debug": ["irrelevant detail"] * 120,
    }
    bloated = {
        "brief": board.brief.model_dump(),
        "board": board.model_dump(),
        "noise": noise,
    }
    stale_write_note = "Simulated stale overwrite: writer_b used revision N while board was already N+1"
    return {
        "bloated_payload_chars": payload_chars(bloated),
        "stale_write_note": stale_write_note,
    }


def inject_event_failure(evented: dict[str, Any]) -> dict[str, Any]:
    log = [ev.model_dump() for ev in evented["event_log"]]
    dropped = {**log[0], "event_type": "research_rdy"}  # typo means missed subscription
    fanout = [f"shadow_subscriber_{i}" for i in range(8)]
    amplification = len(fanout)
    return {
        "dropped_event_type": dropped["event_type"],
        "fanout_subscribers": amplification,
        "duplicate_work_risk": "high",
    }


sm_fail = inject_shared_memory_failure(shared_result)
ev_fail = inject_event_failure(event_result)

print("=== Forced failures ===")
print(f"shared_memory bloated payload chars: {sm_fail['bloated_payload_chars']}")
print(f"shared_memory issue: {sm_fail['stale_write_note']}")
print(f"event_driven dropped event type: {ev_fail['dropped_event_type']}")
print(f"event_driven fanout subscribers: {ev_fail['fanout_subscribers']}")
print(f"event_driven duplicate work risk: {ev_fail['duplicate_work_risk']}")

=== Forced failures ===
shared_memory bloated payload chars: 17819
shared_memory issue: Simulated stale overwrite: writer_b used revision N while board was already N+1
event_driven dropped event type: research_rdy
event_driven fanout subscribers: 8
event_driven duplicate work risk: high


## Reflection Questions

1. Which protocol looked better for your selected task and why?
2. What did you trade off: latency, consistency, or observability?
